# Анализ блочной настройки коэффициентов PID

Поколение v2.x: агент подбирает $(K_p, K_i, K_d)$, но наблюдает не мгновенные сигналы, а статистику ошибки за блок шагов.

Эксперименты:
* `2025-09-30_13-08-38_td3_train_real_async`

In [ ]:
%load_ext autoreload
%autoreload 2

## Описание эксперимента

### Алгоритм
**TD3** (Twin Delayed DDPG), MLP `[256, 256]`. Сбор данных и обучение разделены (асинхронный коллектор). Шум исследования к действиям не подмешивается — разнообразие данных обеспечивает фаза `pretrain` (случайные коэффициенты).

### Окружение (блочная парадигма)
`PidTuningExperimentalEnv`. Один шаг агента = блок из `block_size` сырых шагов; первые `burn_in_steps` отбрасываются, по остальным считается статистика.

- **Состояние** $s$ (2): $[\text{error\_mean\_norm},\ \text{error\_std\_norm}]$ — среднее и СКО ошибки $\text{PV} - \text{setpoint}$ по окну блока, нормированные (÷2.0 и ÷20.0 соответственно).
- **Действие** $a$ (3): $[K_p, K_i, K_d]$ нормированные; в фазе `normal` денормализуются в физические.
- **Награда:** per-step `reward_func` (здесь `exponential`, $k=20$) усредняется по окну блока. В отличие от ранних прогонов, reward **логируется** (в блочной сводке) — восстанавливать не нужно.
- `done` всегда `False` (бесконечный горизонт).

### Фазы
- **warmup** (`warmup_steps`): дефолтный PID (3.5 / 11 / 0.002), форс-границы управления.
- **pretrain** (`pretrain_blocks` блоков): случайные коэффициенты (25–75% диапазона) — предварительный сбор данных, заменяет шум исследования.
- **normal**: действует обученный агент.

### Старт обучения и «switch to NN»
Градиентные шаги начинаются, когда буфер набирает `min_data_for_training` переходов (1 переход = 1 блок). Агент берёт управление (фаза `normal`) в конце pretrain:
$$ \text{neural\_network\_step} = \text{warmup\_steps} + \text{pretrain\_blocks} \cdot \text{block\_size}. $$
`pretrain_blocks` ставится выше `min_data_for_training`, чтобы сеть успела потренироваться на случайных данных до передачи управления. Для 30.09: $1000 + 2655\cdot200 = 532000$ (обучение стартовало ≈ на 2505-м блоке, при `min_data_for_training=2505`).

### Логи окружения
Две разновидности строк: **interaction** (по каждому сырому шагу: PV, CO, границы, коэффициенты) и **блочная сводка** (`block_step=final`: error_mean/std raw+norm, reward). Нормировочные коэффициенты ошибки (2 и 20) в этом прогоне признаны неудачными — планировались 250 / 200.

## Загрузка данных

In [ ]:
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
import re

import pandas as pd
from omegaconf import DictConfig, OmegaConf

from nn_laser_stabilizer.utils.paths import get_experiment_dir
from nn_laser_stabilizer.analysis.sources import parse_keyval_log, parse_train_log


@dataclass
class ExperimentData:
    interaction_df: pd.DataFrame   # по каждому сырому шагу: PV, CO, границы, коэффициенты
    model_df: pd.DataFrame         # блочные сводки: error_mean/std, reward
    train_df: pd.DataFrame         # лоссы обучения
    duration_sec: float
    reward_type: str
    neural_network_step: int       # сырой шаг, на котором агент берёт управление
    config: DictConfig


def _load_env(exp_dir: Path, block_size: int) -> tuple[pd.DataFrame, pd.DataFrame]:
    # env-лог содержит строки двух видов: interaction (есть process_variable) и
    # блочные сводки (есть error_mean, block_step=final). Плюс строки reset (без step).
    raw = parse_keyval_log(exp_dir / "env_logs" / "log.txt")

    interaction_df = raw[raw["process_variable"].notna() & raw["step"].notna()][
        ["step", "time", "phase", "block_step", "kp", "ki", "kd",
         "process_variable", "control_output", "u_min", "u_max"]
    ].reset_index(drop=True)

    model_df = raw[raw["error_mean"].notna()][
        ["step", "phase", "block_step", "kp", "ki", "kd",
         "error_mean", "error_std", "error_mean_norm", "error_std_norm", "reward"]
    ].reset_index(drop=True)
    model_df["block"] = model_df["step"] // block_size

    return interaction_df, model_df


_LOG_TS_RE = re.compile(r"\[(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}),(\d{3})\]")


def _parse_log_time(line: str):
    m = _LOG_TS_RE.match(line)
    if not m:
        return None
    return datetime.strptime(m.group(1), "%Y-%m-%d %H:%M:%S").replace(
        microsecond=int(m.group(2)) * 1000
    )


def _duration_from_main_log(exp_dir: Path) -> float:
    """Длительность эксперимента: 'Training started' -> 'Training finished'."""
    log_path = next(exp_dir.glob("*.log"))
    started = finished = None
    with open(log_path, encoding="utf-8", errors="replace") as f:
        for line in f:
            if "Training started" in line:
                started = _parse_log_time(line)
            elif "Training finished" in line:
                finished = _parse_log_time(line)
    if started is None or finished is None:
        raise ValueError(
            f"В логе {log_path} не найдены метки "
            "'Training started' и/или 'Training finished'"
        )
    return (finished - started).total_seconds()


def load_experiment(
    experiment_name: str, experiment_date: str, experiment_time: str
) -> ExperimentData:
    exp_dir = get_experiment_dir(
        experiment_name=experiment_name,
        experiment_date=experiment_date,
        experiment_time=experiment_time,
    )

    config = OmegaConf.load(exp_dir / ".hydra" / "config.yaml")
    reward_type = OmegaConf.select(config, "env.reward.name", default="absolute")

    block_size = int(OmegaConf.select(config, "env.block_size", default=200))
    warmup_steps = int(OmegaConf.select(config, "env.warmup_steps", default=0))
    pretrain_blocks = int(OmegaConf.select(config, "env.pretrain_blocks", default=0))
    neural_network_step = warmup_steps + pretrain_blocks * block_size

    interaction_df, model_df = _load_env(exp_dir, block_size)

    train_df = parse_train_log(exp_dir / "train_logs" / "log.txt").rename(
        columns={"Loss/Critic": "critic_loss", "Loss/Actor": "actor_loss"}
    )

    duration_sec = _duration_from_main_log(exp_dir)

    return ExperimentData(
        interaction_df=interaction_df,
        model_df=model_df,
        train_df=train_df,
        duration_sec=duration_sec,
        reward_type=reward_type,
        neural_network_step=neural_network_step,
        config=config,
    )

In [ ]:
data = load_experiment(
    experiment_name="td3_train_real_async",
    experiment_date="2025-09-30",
    experiment_time="13-08-38",
)

print(f"Длительность эксперимента: {data.duration_sec / 60:.1f} мин")
print(f"Тип награды: {data.reward_type}")
print(f"Switch to NN (сырой шаг): {data.neural_network_step}")
print(f"interaction_df: {len(data.interaction_df)} строк")
print(f"model_df: {len(data.model_df)} строк")
print(f"train_df: {len(data.train_df)} строк")

In [ ]:
print(OmegaConf.to_yaml(data.config))

In [ ]:
import matplotlib.pyplot as plt
from functools import wraps

OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

if not getattr(plt.savefig, "_auto_wrapped", False):
    _original_savefig = plt.savefig

    @wraps(_original_savefig)
    def _auto_savefig(filename, *args, **kwargs):
        filename = OUTPUT_DIR / Path(filename).name
        return _original_savefig(filename, *args, **kwargs)

    setattr(_auto_savefig, "_auto_wrapped", True)
    plt.savefig = _auto_savefig

## Анализ окружения

In [ ]:
# Фаза для анализа: "warmup" | "pretrain" | "normal".
PHASE = "pretrain"

interaction_phase = data.interaction_df[data.interaction_df["phase"] == PHASE].copy()
model_phase = data.model_df[data.model_df["phase"] == PHASE].copy()

print(f"Фаза {PHASE}: interaction {len(interaction_phase)} строк, model {len(model_phase)} блоков")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(interaction_phase["step"], interaction_phase["process_variable"], color="blue", linewidth=0.8)
ax.axhline(data.config.env.setpoint, color="black", linestyle="--", label="setpoint")
ax.set_title(f"Process Variable ({PHASE})", fontsize=14)
ax.set_xlabel("Step")
ax.set_ylabel("process_variable")
ax.legend()
plt.tight_layout()
plt.savefig("process_variable.pdf")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(interaction_phase["step"], interaction_phase["control_output"], color="red", linewidth=0.8)
ax.set_title(f"Control Output ({PHASE})", fontsize=14)
ax.set_xlabel("Step")
ax.set_ylabel("control_output")
plt.tight_layout()
plt.savefig("control_output.pdf")
plt.show()

In [ ]:
tag = "kp"
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(model_phase["block"], model_phase[tag], color="blue", linewidth=0.8)
ax.set_title(f"{tag} over blocks ({PHASE})", fontsize=14)
ax.set_xlabel("Block")
ax.set_ylabel(tag)
plt.tight_layout()
plt.savefig("kp.pdf")
plt.show()

In [ ]:
tag = "ki"
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(model_phase["block"], model_phase[tag], color="orange", linewidth=0.8)
ax.set_title(f"{tag} over blocks ({PHASE})", fontsize=14)
ax.set_xlabel("Block")
ax.set_ylabel(tag)
plt.tight_layout()
plt.savefig("ki.pdf")
plt.show()

In [ ]:
tag = "kd"
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(model_phase["block"], model_phase[tag], color="green", linewidth=0.8)
ax.set_title(f"{tag} over blocks ({PHASE})", fontsize=14)
ax.set_xlabel("Block")
ax.set_ylabel(tag)
plt.tight_layout()
plt.savefig("kd.pdf")
plt.show()

In [ ]:
# error_mean: сырое и нормированное (norm = raw / 2.0).
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(model_phase["block"], model_phase["error_mean"], color="blue", linewidth=0.8, label="error_mean")
ax.plot(model_phase["block"], model_phase["error_mean_norm"], color="red", linewidth=0.8, label="error_mean_norm")
ax.set_title(f"Error mean ({PHASE})", fontsize=14)
ax.set_xlabel("Block")
ax.set_ylabel("error_mean")
ax.legend()
plt.tight_layout()
plt.savefig("error_mean.pdf")
plt.show()

In [ ]:
# error_std: сырое и нормированное (norm = raw / 20.0).
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(model_phase["block"], model_phase["error_std"], color="blue", linewidth=0.8, label="error_std")
ax.plot(model_phase["block"], model_phase["error_std_norm"], color="red", linewidth=0.8, label="error_std_norm")
ax.set_title(f"Error std ({PHASE})", fontsize=14)
ax.set_xlabel("Block")
ax.set_ylabel("error_std")
ax.legend()
plt.tight_layout()
plt.savefig("error_std.pdf")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(model_phase["block"], model_phase["reward"], color="green", linewidth=0.8)
ax.set_title(f"Reward ({PHASE})", fontsize=14)
ax.set_xlabel("Block")
ax.set_ylabel("reward")
plt.tight_layout()
plt.savefig("reward.pdf")
plt.show()

## Анализ обучения

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
for tag in ["critic_loss", "actor_loss"]:
    if tag in data.train_df.columns:
        ax.plot(data.train_df["step"], data.train_df[tag], label=tag)
ax.set_title("Training loss", fontsize=14)
ax.set_xlabel("Step")
ax.set_ylabel("Value")
ax.legend(title="Metric")
plt.tight_layout()
plt.savefig("train_loss.pdf")
plt.show()

## График для ВКВО

In [ ]:
BLOCK_RANGE = (2400, 2450)

zoom = model_phase[(model_phase["block"] >= BLOCK_RANGE[0]) & (model_phase["block"] <= BLOCK_RANGE[1])]

corr_kp_std = zoom["kp"].corr(zoom["error_std"])
print(f"Корреляция Kp ↔ error_std (блоки {BLOCK_RANGE[0]}–{BLOCK_RANGE[1]}): {corr_kp_std:.4f}")

> **Замечание по оси времени.** Ось X переведена в секунды через `STEP_INTERVAL_SEC = 0.005` (5 мс на шаг, как в исходном анализе): 1 блок = `block_size · 0.005` = 1 с. Отображаемое время носит приблизительный характер.

In [ ]:
from matplotlib.gridspec import GridSpec
from matplotlib.ticker import MultipleLocator

plt.rcParams.update({
    "font.size": 14,
    "axes.labelsize": 16,
    "axes.titlesize": 18,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 14,
})

SETPOINT = data.config.env.setpoint
BLOCK_SIZE = data.config.env.block_size
STEP_INTERVAL_SEC = 0.005 

model_time_sec = model_phase["block"] * BLOCK_SIZE * STEP_INTERVAL_SEC
TIME_RANGE = (
    BLOCK_RANGE[0] * BLOCK_SIZE * STEP_INTERVAL_SEC,
    BLOCK_RANGE[1] * BLOCK_SIZE * STEP_INTERVAL_SEC,
)

fig = plt.figure(figsize=(20, 8))
gs = GridSpec(2, 2, figure=fig, width_ratios=[1, 1], height_ratios=[1, 1])

# Левый верхний: СКО ошибки.
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(model_time_sec, model_phase["error_std"], color="red", label="СКО ошибки")
ax1.set_ylabel("СКВО ошибки", color="red")
ax1.tick_params(axis="y", labelcolor="red")
ax1.set_xlim(*TIME_RANGE)
ax1.set_ylim(0, 200)
ax1.legend(loc="best")
ax1.xaxis.set_minor_locator(MultipleLocator(5))
ax1.grid(True, axis="x", which="both", linestyle="--", alpha=0.4)

# Левый нижний: Kp.
ax2 = fig.add_subplot(gs[1, 0], sharex=ax1)
ax2.plot(model_time_sec, model_phase["kp"], color="blue", label=r"$K_{\text{p}}$")
ax2.set_xlabel("Время (с)")
ax2.set_ylabel(r"$K_{\text{p}}$", color="blue")
ax2.tick_params(axis="y", labelcolor="blue")
ax2.set_ylim(4, 13)
ax2.legend(loc="best")
ax2.xaxis.set_minor_locator(MultipleLocator(5))
ax2.grid(True, axis="x", which="both", linestyle="--", alpha=0.4)

# Правый: process variable.
ax3 = fig.add_subplot(gs[:, 1])
pv_zoom = interaction_phase[
    (interaction_phase["step"] // BLOCK_SIZE >= BLOCK_RANGE[0])
    & (interaction_phase["step"] // BLOCK_SIZE <= BLOCK_RANGE[1])
]
ax3.plot(pv_zoom["step"] * STEP_INTERVAL_SEC, pv_zoom["process_variable"],
         color="green", label=r"$U_{\text{АЦП}}$")
ax3.axhline(SETPOINT, color="black", linestyle="--")
ax3.set_ylabel("Напряжение на фотодетекторе (АЦП)", color="green")
ax3.set_xlabel("Время (с)")
ax3.set_xlim(*TIME_RANGE)
ax3.set_ylim(1000, 1500)
ax3.tick_params(axis="y", labelcolor="green")
ax3.legend(loc="best")
ax3.xaxis.set_minor_locator(MultipleLocator(5))
ax3.grid(True, axis="x", which="both", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.savefig("kp_split_combo.png", dpi=300, bbox_inches="tight")
plt.show()